# Axis 3 — External validity (`50_external_crosscheck`)

The final axis: reproduce results **published elsewhere, on the same data**, to
catch whole-package convention/scale errors that Axes 1–2 (internally
self-consistent) cannot see (TESTING.md §9).

Two anchor datasets + a scale-bridging check:

1. **Verbal Aggression** (De Boeck & Partchev 2012, *JSS*) — the canonical IRTree
   benchmark. Validates the **tree core** against an independent estimator (R `lme4`
   `glmer`, logit link).
2. **PISA 2022 disengagement** (Mehou, Leventhal, Hunsberger & Bao 2026) — validates
   the **full feature set**: 2PL/GPCM ability node, examinee covariates on
   disengagement, estimated latent correlation. Target: the published Table 3.
3. **Probit→logit scale bridging** — both references use the **logit** link; the
   probit fits are converted with `fit.to_logit()` (D = 1.702). The factor goes on
   whichever quantity carries the latent scale: on dimensions whose nodes all have
   slopes (σ² = 1), probit slopes ×1.702 and everything inside a(θ − b + xβ)
   (θ, b, thresholds, β) is unchanged; on Rasch-type dimensions θ, b and β ×1.702.
   ρ is always unchanged.

> **Paper materials.** The verbal-aggression comparison with lme4 and mirt used in
> the JSS paper (Section 7.2) is `replication/verbagg/72_agreement.ipynb`; this
> notebook keeps the test-suite version. Section 2 below is the PISA analysis of
> paper Section 6.2.

> **You supply the data.** The De Boeck data is public (`irtrees` R package,
> `VerbAgg3`); the PISA data is extracted by you from the public-use PISA 2022
> files (responses, response times and questionnaire scales). Cells that need
> your data are marked **`# >>> SUPPLY`**. Everything else runs as written.


## Setup


In [ ]:
import sys, pathlib
!{sys.executable} -m pip install -e "C:\Users\manonfi\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\PACKAGE\Irtree"

In [ ]:
import sys, warnings, pathlib, numpy as np
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec as ts
import irtree.data_prep as dp
import irtree.node_models as nm
import irtree.priors as pr
import irtree.fit as ft
import irtree.ingest as ing
import pandas as pd

# External validity compares package estimates to KNOWN published points.
# Print each estimate next to its published target and judge for yourself
# (no pass/fail asserts -- same spirit as the recovery notebooks 40-42).
def show(label, est, target=None, hpd=None):
    line = f'  {label:26s} est {est:+.4f}'
    if target is not None:
        line += f'   published {target:+.4f}   diff {est - target:+.4f}'
    if hpd is not None:
        line += f'   95% HPD [{hpd[0]:+.4f}, {hpd[1]:+.4f}] {"inside" if hpd[0] <= est <= hpd[1] else "OUTSIDE"}'
    print(line)

ROOT = pathlib.Path.cwd().parents[1]
import importlib.util
harness_path = ROOT / "tests" / "_harness.py"
spec = importlib.util.spec_from_file_location("_harness", harness_path)
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)

# ---- probit -> logit conversion for the hand-made check of Section 3 ----
# For package fits use fit.to_logit(): the factor depends on how each dimension is
# identified (sigma2 = 1 with slopes: slopes x1.702, theta/b/beta unchanged;
# Rasch-type: theta, b, beta x1.702).  This helper covers the 2PL case only.
D_PL = 1.702
def to_logit(value, role):
    """2PL dimension (sigma2 = 1): role 'slope' -> x1.702; 'location'/'correlation' unchanged."""
    if role == 'slope':        return D_PL * np.asarray(value)
    if role in ('location','correlation','difficulty'): return np.asarray(value)
    raise ValueError(role)

# ---- runtime log ----
import time, contextlib
RUNTIME = []
@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter(); yield; dt = time.perf_counter() - t0
    RUNTIME.append({'step': step, 'seconds': round(dt, 1)})
    print(f'[{step}] ' + (f'{dt/60:.1f} min' if dt >= 60 else f'{dt:.1f} s'))

## 1. Verbal Aggression — the linear response tree (De Boeck & Partchev 2012)

**The application** (their §2.4, *"Are the categories ordinal?"*): 316 subjects,
24 items, 3 categories `0=no, 1=perhaps, 2=yes`. Linear tree, 2 nodes:

```
  node Y*1 (admission):  'no'      vs  'perhaps or yes'
  node Y*2 (affirmation): 'perhaps' vs  'yes'   (only if not 'no')
```

Mapping `cbind(c(0,1,1), c(NA,0,1))` — cat 0→(0,NA), cat 1→(1,0), cat 2→(1,1).

**The comparison of interest** is the *ordinality test*: fit a **1-dimensional**
model (one trait at both nodes) and a **2-dimensional** model (a trait per node,
correlation estimated). De Boeck's published result (R `glmer`, logit):

- 2D fits better than 1D (AIC 12378 vs 12778; BIC 12751 vs 13137)
- **latent correlation between the two traits = 0.392**

Node model: **Rasch (1PL)** — De Boeck's GLMM *cannot* estimate discrimination
(2PL/3PL are excluded from the `glmer` family), so it is `item:node` fixed
difficulties + random person effects.


### 1.1 Build the exact De Boeck tree (both 1D and 2D)

The 1D vs 2D distinction is `node_to_latent`: `(1,1)` = both nodes share one
trait (1D); `(1,2)` = a separate trait per node (2D).


In [ ]:
M_va = np.array([[0, np.nan],
                 [1, 0   ],
                 [1, 1   ]])   # rows: no/perhaps/yes ; cols: Y*1, Y*2

spec_va_1d = ts.IRTreeSpec(M_va, node_to_latent=(1,1),
                           category_names=('no','perhaps','yes'),
                           node_names=('admission','affirmation'), values_start=0)

spec_va_2d = ts.IRTreeSpec(M_va, node_to_latent=(1,2),
                           category_names=('no','perhaps','yes'),
                           node_names=('admission','affirmation'), values_start=0)



### 1.2 Load the data 

Get `VerbAgg3` from the R `irtrees` package (wide form, 316 × 24, values 0/1/2),
export to CSV, and load it here as an integer array `Y_va` of shape `[316, 24]`.

```r
# in R:
library(irtrees); data(VerbAgg3)
write.csv(VerbAgg3[, -(1:2)], 'verbagg3.csv', row.names = FALSE)
```


In [ ]:
# VerbAgg3 responses (0/1/2), shape [316, 24]
Y_va = np.loadtxt(r"C:\Users\mehoujm\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\Publication work\Verbal agression rep\verbagg3.csv", delimiter=',', skiprows=1).astype(int)
print('  Y_va shape', Y_va.shape, 'categories', np.unique(Y_va))


In [ ]:
Y_va[:3,]

### 1.3 Probit fit (Gibbs) — 1D vs 2D, recover ρ

Probit nodes use Albert–Chib Gibbs. Fit the 1-dimensional and 2-dimensional trees, compare on WAIC, and read the latent correlation ρ against De Boeck's published 0.392.

In [ ]:
# Route the raw response matrix through irtree.ingest (the id-joined front
# door over expand_data). Wide layout: one row per person, item ids = headers.
va_df = pd.DataFrame(Y_va, columns=[f'item_{i+1}' for i in range(Y_va.shape[1])])
va_df.insert(0, 'person_id', [f'p{p+1}' for p in range(Y_va.shape[0])])
VA_START = int(np.nanmin(Y_va))     # VerbAgg3 export: 0/1/2 -> 0 (1/2/3 -> 1)
data_1d = ing.load_data(va_df, spec=spec_va_1d, categories_start=VA_START, verbose=False)
data_2d = ing.load_data(va_df, spec=spec_va_2d, categories_start=VA_START, verbose=False)

In [ ]:
data_1d.Y_star[0]

In [ ]:
# node models — probit (Albert–Chib Gibbs at each node)
mods = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]

In [ ]:
# Priors specification
pri_2d = pr.build_default_priors(spec_va_2d, mods)
# theta.correlation: LKJ(1) is the package default (estimated)
pri_2d.set('b',  pr.NormalPrior(mean=0, variance = 5))  # estimate rho

In [ ]:
N_IT, N_CH = 10000, 1   # evidence-mode warmup is the default; no n_warmup to set

with timed('VerbAgg probit 1D'):
    fit_1d = ft.irtree_fit(data_1d, spec_va_1d, mods,              n_iter=N_IT, n_chains=N_CH, seed=1)
with timed('VerbAgg probit 2D'):
    fit_2d = ft.irtree_fit(data_2d, spec_va_2d, mods, priors=pri_2d, n_iter=N_IT, thin_by=1, n_chains=N_CH, seed=1)

def waic_val(w): return next(v for k, v in vars(w).items() if 'waic' in k.lower() or 'elpd' in k.lower())
w1, w2 = fit_1d.waic(), fit_2d.waic()
print(f'  WAIC   1D = {waic_val(w1):.1f}    2D = {waic_val(w2):.1f}    (2D lower => ordinality supported)')

names = fit_2d.results[0].param_names
ridx  = [j for j, n in enumerate(names) if 'rho' in n or ('corr' in n and 'matrix' not in n)]
ch    = np.concatenate([np.asarray(r.chain, np.float64) for r in fit_2d.results])
rho_probit = float(ch[:, ridx[0]].mean())
show('latent rho (probit 2D)', rho_probit, target=0.392)   # De Boeck 2012; scale-free

In [ ]:
fit_1d.report(path = r"C:\Users\mehoujm\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\Publication work\Verbal agression rep\1d_verb_agg_irtree_report.html", plot_params = 2 )

In [ ]:
fit_2d.report(path = r"C:\Users\mehoujm\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\Publication work\Verbal agression rep\2d_verb_agg_irtree_report.html", plot_params = 2)

### 1.4 Logit fit — native-logit refit of the 2D model

The same 2D model fit from scratch under the logit link (MH at each node). ρ is scale-free, so the logit fit should land on the same correlation as the probit fit above — a controlled check of the link, not a new result.

In [ ]:
# node models — logit (MH at each node)
mods_logit = [nm.LogitRaschNode(), nm.LogitRaschNode()]

pri_2d_logit = pr.build_default_priors(spec_va_2d, mods_logit)
pri_2d_logit.set('b', pr.NormalPrior(mean=0, variance=5))

with timed('VerbAgg logit 2D'):
    fit_2d_logit = ft.irtree_fit(data_2d, spec_va_2d, mods_logit, priors=pri_2d_logit,
                                 n_iter=N_IT, n_chains=N_CH, seed=1)

names_l = fit_2d_logit.results[0].param_names
ridx_l  = [j for j, n in enumerate(names_l) if 'rho' in n or ('corr' in n and 'matrix' not in n)]
ch_l    = np.concatenate([np.asarray(r.chain, np.float64) for r in fit_2d_logit.results])
rho_logit = float(ch_l[:, ridx_l[0]].mean())

# logit nodes are MH -- evidence-mode warmup reports where each block froze:
print('  MH accept rates:', {k: round(v, 3) for k, v in fit_2d_logit.results[0].accept_rates.items()})
print('  froze at       :', fit_2d_logit.results[0].frozen_at)
show('latent rho (logit 2D)', rho_logit, target=0.392)          # vs De Boeck
show('logit vs probit rho',   rho_logit, target=rho_probit)     # scale-free: should match

> **NOTE — item difficulties.** De Boeck 2012 reports only ρ and the AIC/BIC
> ordering, not the 48 per-item difficulties. To compare difficulties per item,
> fit the `glmer` models in R (`onedim`/`twodim` from their §2.4) yourself, read
> off the `item:node` fixed effects, convert this package's probit b's to the
> logit scale (**b is a location → unchanged**, only `a`/`theta` scale by 1.702),
> and drop them into the table below. Left as a TODO cell.


In [ ]:
# >>> SUPPLY (optional): De Boeck glmer item:node difficulties, to compare per-item.
# b_deboeck_logit = np.array([...])   # from R  (already on logit scale)
# b_here_probit   = ch_1d_or_2d_b_columns.mean(0)
# b_here_logit    = to_logit(b_here_probit, 'location')   # locations: unchanged
# compare b_here_logit vs b_deboeck_logit (expect agreement within MCMC + link error)
print('  (item-difficulty comparison: supply the R glmer estimates to enable)')


## 2. PISA 2022 disengagement — the full feature set (Mehou et al. 2026)

**The model** (your paper, Eqs. 2–5):

- **Tree**: 2 nodes. Node 1 = rapid-guess (disengagement θ⁽¹⁾); node 2 = ability θ⁽²⁾.
  Dichotomous items → 3 categories (RG / incorrect / correct); polytomous items →
  4 categories (RG / no-credit / partial / full).
- **Node 1**: 1PL, a **single shared discrimination a⁽¹⁾** across items, logit link,
  with the **covariate regression** on θ⁽¹⁾ (Eq. 5): 9 covariates, 10 coefficients
  (grade enters as two dummies).
- **Node 2**: **2PL** for the 105 dichotomous items, **GPCM** for the 10 polytomous
  items (mixed-format), logit link.
- **Estimated latent correlation** ρ(θ⁽¹⁾, θ⁽²⁾).

**Covariate order (β₀…β₉):** intercept, Female, Language-of-test, ESCS, Growth
Mindset, **Grade-10 dummy**, **Grade-9 dummy** (11th grade = reference),
Education-expectation, Sense-of-belonging, Family-support, Bullying.

**Priors (your Table 1):** b ~ N(0, σ²=5); a ~ LogNormal(0, √2); c ~ N(0.25, …);
β ~ N(0, σ²=4); ρ ~ N(0, √2) truncated to (−1, 1); θ ~ MVN(0, Σ).

**In irtree:** b ~ N(0, 5) and β ~ N(0, 4) are set to match; ρ uses the package
default LKJ(1) (uniform on (−1, 1)); the shared node-1 slope uses the default
TruncNormal(1, 0.5) on (0, ∞) at the probit node (exact Gibbs draw). Both node-1
and node-2 dimensions have slopes, so their variances are fixed at 1, as in the
published model. With 2,201 examinees these prior differences should matter little.


> **Two honest caveats, both from you:**
> 1. **Missing covariates.** Your paper imputed them with *factorized-equations*
>    imputation; this package draws them as **latent parameters each iteration**.
>    These are different (both defensible) treatments, so the PISA reproduction is
>    *"same model, different missing-data handling"* — not a decimal-exact replica.
>    Expect close, not identical.
> 2. **Rapid-guess derivation.** Building the observed `Y` (RG vs incorrect/correct)
>    from response times via Normative Thresholds is your upstream step; the notebook
>    consumes the derived `Y`.


### 2.1 The published targets (Table 3) — the golden column


In [ ]:
import pandas as pd

path = r"C:\Users\mehoujm\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\Publication work\nested_models_dataset_with_missing_imputed_2pl.csv"
data = pd.read_csv(path).apply(pd.to_numeric, errors="coerce")

I = data.loc[:, "I1":"I115"]
RG = data.loc[:, "RG1":"RG115"]

# category coding: 0 = rapid guess, then score + 1 (1 = incorrect / no credit,
# 2 = correct / partial, 3 = full credit); planned-missing cells stay NaN.
# -> the categories start at 0 (categories_start=0 below).
representative = I.add(1).mask(RG.values == 1, 0)
covariates = data.iloc[:, 231:]
print('response categories:', np.unique(representative.stack().to_numpy()))
print('covariates:', list(covariates.columns))
print('missing values per covariate:'); print(covariates.isna().sum())

In [ ]:
covariates

In [ ]:
# published EAP / SD / 95% HPD, on the LOGIT scale (your paper's link)
PISA_TARGETS = {
    'a1'          : (2.9570, 0.1066, (2.7555, 3.1700)),
    'beta0_intcpt': (0.8118, 0.1857, (0.4648, 1.1864)),
    'beta1_female': (-0.2480, 0.0569, (-0.3605, -0.1371)),
    'beta2_lang'  : (0.1411, 0.0721, (0.0030, 0.2872)),
    'beta3_escs'  : (-0.0277, 0.0304, (-0.0884, 0.0305)),
    'beta4_grow'  : (0.0001, 0.0434, (-0.0855, 0.0863)),
    'beta51_gr10' : (-1.2738, 0.0808, (-1.4307, -1.1144)),
    'beta52_gr9'  : (0.5338, 0.0992, (0.3438, 0.7349)),
    'beta6_exped' : (0.0169, 0.0795, (-0.1452, 0.1670)),
    'beta7_belong': (-0.0550, 0.0528, (-0.1581, 0.0482)),
    'beta8_famsup': (0.0433, 0.0557, (-0.0703, 0.1514)),
    'beta9_bully' : (-0.0224, 0.0533, (-0.1283, 0.0780)),
    'rho'         : (-0.5223, 0.0308, (-0.5788, -0.4588)),
}
for k,(m,s,hpd) in PISA_TARGETS.items():
    print(f'  {k:14s} EAP {m:+.4f}  SD {s:.4f}  95% HPD [{hpd[0]:+.4f}, {hpd[1]:+.4f}]')


### 2.2 Supply the derived data  `# >>> SUPPLY`

Three arrays, all for the 2,201 U.S.A. examinees:
- `Y_pisa` `[2201, 115]` — observed categories per your rapid-guess derivation:
  `1=RG, 2=incorrect, 3=correct` for dichotomous items; `1=RG, 2=no-credit,`
  `3=partial, 4=full` for polytomous. Planned-missing cells (unassigned clusters)
  as your missing code.
- `X_pisa` `[2201, 9]` — covariates in the β₁…β₉ order above (missing = your code).
- `poly_items` — a length-115 boolean (or index list): True for the 10 polytomous items.


In [ ]:
Y_pisa     = representative.to_numpy()      # [2201, 115], see coding above
# Center the covariates (the trait mean is fixed at 0; paper Section 4). Centering
# leaves the slopes beta unchanged; it moves the intercept, which is not estimated
# here (the item locations absorb it). Missing values stay NaN and are drawn as
# parameters each iteration with prior N(0, 1) on the centered scale.
X_names    = list(covariates.columns)
X_pisa     = (covariates - covariates.mean()).to_numpy()   # [2201, 10], beta1..beta9 order
assert X_pisa.shape[1] == 10, f'expected 10 covariate columns (beta1..beta9 with two grade dummies), got {X_pisa.shape[1]}'
poly_items = [0] * 105 + [1] * 10 # bool[115], True for the 10 GPCM items

### 2.3 Build the model and fit  (runs only when PISA data is supplied)

Node 2 is **mixed-format**: a GPCM node whose `pseudo_item_outcomes` marks which
items are dichotomous (2 outcomes → reduces to 2PL) vs polytomous (3 outcomes).
All covariates enter **node 1** as person covariates; ρ is estimated (the
package default, LKJ(1)).

**Settings and memory.** Each chain stores every retained draw of every parameter
(about 4,700 here, mostly the 2 × 2,201 traits) in float32: 10,000 retained draws
per chain take about 190 MB. `thin_by` keeps every k-th draw. Four chains run in
parallel processes and give R-hat; one chain of 100,000 iterations (the earlier
setting) stores about 1.9 GB.


In [ ]:
np.sum(Y_pisa[:,-1] == 0)

In [ ]:
# node-2 mixed structure: dichotomous science items score 0/1 (2 cats at node 2),
# polytomous score 0/1/2 (3 cats). pseudo_item_outcomes: 2 for dich, 3 for poly.
n_items = Y_pisa.shape[1]
pio = np.where(np.asarray(poly_items), 3, 2)


In [ ]:
# The disengagement tree with a polytomous ability node: 4 categories
# (0 rapid guess, 1 no credit, 2 partial / correct, 3 full credit).
spec_pisa = ts.disengagement_tree(binary=False)
spec_pisa.mapping

In [ ]:
# ---- settings ----
N_ITER_PISA, N_CHAINS_PISA, THIN_PISA = 20000, 4, 4     # 4 x 5,000 retained draws
PISA_START = 0                                          # categories coded from 0 (see 2.2)

# Route PISA through irtree.ingest: responses wide + the person covariates
# joined on person_id (ingest's separate-table mode -- the join is its whole job).
resp_pisa = pd.DataFrame(Y_pisa, columns=[f'item_{i+1}' for i in range(Y_pisa.shape[1])])
resp_pisa.insert(0, 'person_id', [f'p{p+1}' for p in range(Y_pisa.shape[0])])
xcov_pisa = pd.DataFrame(X_pisa, columns=X_names)
xcov_pisa.insert(0, 'person_id', resp_pisa['person_id'])
data_pisa = ing.load_data(resp_pisa, spec=spec_pisa, person_covariates=xcov_pisa,
                          categories_start=PISA_START, verbose=False)
import irtree
print(irtree.summarize_prepared(data_pisa, spec_pisa))

# --- PROBIT model: node 1 with ONE discrimination a(1) shared by all items (paper
#     Eq. 2) and every covariate; node 2 mixed 2PL/GPCM (the GPCM is a logit model,
#     shared by the probit and logit fits).
node1_probit  = nm.ProbitTwoPLNode(common_discrimination=True, person_covariates=True)
node2         = nm.GPCMNode(pseudo_item_outcomes=pio)     # 2PL(dich)/GPCM(poly)
models_probit = [node1_probit, node2]

priors = pr.build_default_priors(spec_pisa, models_probit)   # theta.correlation: LKJ(1)
priors.set('b',    pr.NormalPrior(mean=0.0, variance=5.0))
priors.set('beta', pr.NormalPrior(mean=0.0, variance=4.0))
priors.summary()

with timed('PISA probit fit'):
    fit_pisa = ft.irtree_fit(data_pisa, spec_pisa, models_probit, priors=priors,
                             n_iter=N_ITER_PISA, n_chains=N_CHAINS_PISA,
                             thin_by=THIN_PISA, seed=42)

In [ ]:
fit_pisa.diagnostics()

### 2.4 Compare to Table 3  (probit → logit conversion applied)

Pass criteria, in decreasing strictness (§9.2): (1) sign/significance of every
covariate matches; (2) each converted EAP inside the published HPD; (3) ρ within
~0.05 of −0.522; (4) a⁽¹⁾ near 2.96.

`fit.to_logit()` does the conversion. Both dimensions have slopes (σ² = 1), so the
probit slope a⁽¹⁾ is multiplied by 1.702, while β (inside a(θ − b + xβ), in θ
units) and ρ are unchanged. *(The earlier version of this cell multiplied β by
1.702 too, which is wrong for this parameterization.)*


In [ ]:
lg = fit_pisa.to_logit()                 # logistic-equivalent summary, factor per row
print(*lg.attrs.get('notes', []), sep='\n')
beta_rows = [n for n in lg.index if n.startswith('beta_1_')]   # beta1..beta9, X_names order
keys_beta = ['beta1_female','beta2_lang','beta3_escs','beta4_grow','beta51_gr10',
             'beta52_gr9','beta6_exped','beta7_belong','beta8_famsup','beta9_bully']
assert len(beta_rows) == len(keys_beta), beta_rows

print('PROBIT fit on the logistic-equivalent scale (fit.to_logit):')
for k, n in zip(keys_beta, beta_rows):
    pub, _, hpd = PISA_TARGETS[k]
    show(f'{k} [{n}]', lg.loc[n, 'mean'], target=pub, hpd=hpd)
pa, _, phpd = PISA_TARGETS['a1']
show('a1 (x1.702)', lg.loc['a_1_1', 'mean'], target=pa, hpd=phpd)   # shared slope: a_1_1 = a_1_i
prho, _, rhpd = PISA_TARGETS['rho']
show('rho (scale-free)', lg.loc['rho_1_2', 'mean'], target=prho, hpd=rhpd)
lg.loc[beta_rows + ['a_1_1', 'rho_1_2'], ['mean', 'sd', 'hdi_lo', 'hdi_hi', 'factor']].round(4)

### 2.5 Logit fit — native-logit node 1 (compares to Table 3 directly)

Node 1 refit under the logit link (node 2 GPCM is already logit). Table 3 is published on the logit scale, so this fit compares to it **directly, with no ×1.702 conversion** — the complement to the converted probit fit above. This is a second full-length PISA run with the same settings (`N_ITER_PISA`, `N_CHAINS_PISA`, `THIN_PISA`).

In [ ]:
# --- LOGIT model: node 1 logit with one shared discrimination a(1) and every
# covariate (paper Eq. 2); node 2 the same mixed 2PL/GPCM node.
node1_logit  = nm.LogitTwoPLNode(common_discrimination=True, person_covariates=True)
models_logit = [node1_logit, node2]

priors_l = pr.build_default_priors(spec_pisa, models_logit)   # theta.correlation: LKJ(1)
priors_l.set('b',    pr.NormalPrior(mean=0.0, variance=5.0))
priors_l.set('beta', pr.NormalPrior(mean=0.0, variance=4.0))

with timed('PISA logit fit'):
    fit_pisa_logit = ft.irtree_fit(data_pisa, spec_pisa, models_logit, priors=priors_l,
                                   n_iter=N_ITER_PISA, n_chains=N_CHAINS_PISA,
                                   thin_by=THIN_PISA, seed=42)

sl = fit_pisa_logit.summary_df()          # already logistic: compare directly
print('LOGIT fit, compared to Table 3 DIRECTLY (no conversion):')
print('  node-1 MH froze at:', fit_pisa_logit.results[0].frozen_at)
beta_rows = [n for n in sl.index if n.startswith('beta_1_')]
for k, n in zip(keys_beta, beta_rows):
    pub, _, hpd = PISA_TARGETS[k]
    show(f'{k} [{n}]', sl.loc[n, 'mean'], target=pub, hpd=hpd)
pa, _, phpd = PISA_TARGETS['a1']
show('a1', sl.loc['a_1_1', 'mean'], target=pa, hpd=phpd)
prho, _, rhpd = PISA_TARGETS['rho']
show('rho (scale-free)', sl.loc['rho_1_2', 'mean'], target=prho, hpd=rhpd)

## 3. Probit → logit scale bridging (self-contained, runs now)

The conversion the two comparisons above rely on. Fit a probit model here,
convert (slopes ×1.702, locations unchanged), and confirm it matches a
from-scratch **logit** fit of the *same data* — the controlled check that isolates
the conversion (§9.3). No external data needed.


In [ ]:
from scipy.special import ndtr
from scipy.stats import norm

# simulate a 2PL under PROBIT, fit under both links, compare a and b
rng = np.random.default_rng(0); P, I = 4000, 30
theta = rng.normal(0,1,P); a_pr = rng.uniform(0.8,1.6,I); b_pr = rng.normal(0,1,I)
Z = a_pr[None,:]*(theta[:,None]-b_pr[None,:])
Yb = (rng.random((P,I)) < ndtr(Z)).astype(int)

def irls_2pl(y, theta):
    beta = np.array([1.0, 0.0]); X = np.column_stack([theta, -np.ones_like(theta)])
    for _ in range(100):
        eta = X@beta; mu = 1/(1+np.exp(-eta)); d = np.clip(mu*(1-mu),1e-9,1)
        z = eta + (y-mu)/d
        beta = np.linalg.solve((X*d[:,None]).T@X + 1e-8*np.eye(2), (X*d[:,None]).T@z)
    return beta[0], beta[1]/beta[0]   # a_logit, b_logit
def irls_2pl_probit(y, theta):
    beta = np.array([1.0, 0.0]); X = np.column_stack([theta, -np.ones_like(theta)])
    for _ in range(100):
        eta = X@beta; mu = np.clip(ndtr(eta),1e-9,1-1e-9); ph = norm.pdf(eta)
        w = ph**2/(mu*(1-mu)); z = eta + (y-mu)/np.clip(ph,1e-9,1)
        beta = np.linalg.solve((X*w[:,None]).T@X + 1e-8*np.eye(2), (X*w[:,None]).T@z)
    return beta[0], beta[1]/beta[0]

a_log = np.array([irls_2pl(Yb[:,i], theta)[0] for i in range(I)])
a_pro = np.array([irls_2pl_probit(Yb[:,i], theta)[0] for i in range(I)])
b_log = np.array([irls_2pl(Yb[:,i], theta)[1] for i in range(I)])
b_pro = np.array([irls_2pl_probit(Yb[:,i], theta)[1] for i in range(I)])

a_ratio = np.median(a_log/a_pro)
b_diff  = np.median(np.abs(b_log - b_pro))
print(f'  discrimination a:  logit/probit ratio = {a_ratio:.3f}   (expect ~1.702)')
print(f'  difficulty b:      |logit - probit|    = {b_diff:.3f}   (expect ~0, locations unchanged)')
print(f'  converting probit a by 1.702 -> matches logit a to {np.median(np.abs(to_logit(a_pro,"slope")-a_log)):.3f}')

## Summary

Probit and logit are run as separate, explicit analyses in each comparison, so you can line them up yourself:

- **Verbal Aggression** validates the tree core. Probit fit: 2D beats 1D on WAIC and ρ≈0.39 reproduces De Boeck's independent (logit `glmer`) estimate. Logit refit: same ρ (scale-free), confirming the link is not driving the result.
- **PISA** validates the full feature set against your published Table 3. The probit fit is converted with `fit.to_logit()` (probit slope ×1.702; β and ρ unchanged on these σ² = 1 dimensions); the native-logit fit compares to Table 3 directly with no conversion. The documented missing-data caveat applies (latent-parameter draws vs factorized imputation): expect close, not identical.
- **Scale bridging** (self-contained) confirms the conversion on a 2PL dimension: slopes ×1.702, locations and ρ unchanged.

External validity is a spot-check against known points, not a proof over the whole model space (Axes 1–2 cover that). It catches the convention/scale errors that are invisible from inside the package — but only where someone has already been.

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {'step': 'TOTAL', 'seconds': rt['seconds'].sum()}
rt